In [3]:
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import pyrubberband as pyrb
from scipy import interpolate
from pydub import AudioSegment
from sklearn.cluster import KMeans
from IPython.display import Audio
import scipy
from essentia.standard import *
import essentia.streaming as ess
import essentia.standard as es

path1 = '/users/seanmullins333/desktop/Training Data/Calvin Harris/One Kiss.wav'
path2 = '/users/seanmullins333/desktop/Training Data/Calvin Harris/Heatstroke.wav'
y, sr = librosa.load(path1)

/var/folders/v8/l9k84q0j5dvb69nc7wjtd5x80000gn/T/ipykernel_18855/505663360.py:17: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path1)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [4]:
class effects():
    
    def transition_points(sample):
        y=sample
        tempo, beats = librosa.beat.beat_track(y=y, sr=22050)

        beat_times = [round(i,3) for i in librosa.frames_to_time(beats)]
        beat_df = pd.DataFrame(beat_times)

        #Create a lowpass RMS feature to add to our cluster
        cutoff_hz = 70
        nyquist = 0.5 * 22050
        normal_cutoff = cutoff_hz / nyquist
        filter_order = 8
        b, a = scipy.signal.butter(filter_order, normal_cutoff, btype='low')
        y_low = scipy.signal.lfilter(b, a, y)

        #Create a percussive layer
        y_perc = librosa.effects.percussive(y, margin = 5)
        y_perc = scipy.signal.lfilter(b, a, y_perc)


        #Set duration for RMs to calculate. In this case we are doing .01
        window_duration = 0.05  # 100 milliseconds
        frame_length = int(window_duration * sr)  # Convert duration to samples
        hop_length = int(frame_length / 2)  # Choose a smaller hop length for overlap (e.g., half of frame length)


        #Create the cluster features
        rms_cluster = pd.DataFrame(librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0])
        rms_cluster = rms_cluster.rename(columns={0:'RMS'})
        rms_cluster['SMA'] = rms_cluster.rolling(150, center=True).mean().fillna(0)

        rms_cluster['low'] = pd.DataFrame(librosa.feature.rms(y=y_low, frame_length=frame_length, hop_length=hop_length)[0])
        rms_cluster['SMA_low'] = rms_cluster.low.rolling(150, center=True).mean().fillna(0)

        rms_cluster['perc'] = pd.DataFrame(librosa.feature.rms(y=y_perc, frame_length=frame_length, hop_length=hop_length)[0])
        rms_cluster['SMA_perc'] = rms_cluster.perc.rolling(40, center=True).mean().fillna(0)



        rms_cluster['PCT'] = rms_cluster.RMS.pct_change()
        rms_cluster['completion'] = rms_cluster.index / len(rms_cluster.RMS)


        #Run a KMeans clusting to find transition points

        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        X_scaled = rms_cluster[['SMA_low', 'SMA_perc', 'SMA', 'low']]
        X_scaled = scaler.fit_transform(X_scaled)

        feature_weights = [1, .4, 1, .2]  # Adjust weights as needed
        X_weighted = X_scaled * feature_weights


        num_clusters = 2
        kmeans = KMeans(n_clusters=num_clusters, random_state=42)
        sma_array = np.array(X_weighted)
        clusters = kmeans.fit_predict(sma_array)

        clusters = clusters.reshape(-1, 1)

        rms_cluster['clusters'] = clusters
        cue = [0]
        duration = 1 / 22050

        timestamps = [i * duration for i in range(len(rms_cluster.index))]

        for i,j in zip(rms_cluster.clusters, rms_cluster.clusters[1:]):
            if i != j:
                cue.append(1)
            else:
                cue.append(0)

        rms_cluster['cue'] = cue        
        rms_cluster['seconds'] = round(rms_cluster.completion * librosa.get_duration(y=y),2)


        return rms_cluster[rms_cluster.cue == 1]


    def beat_matching(path):
        audio = MonoLoader(filename=path)()

        # Compute beat positions and BPM
        rhythm_extractor = RhythmExtractor2013(method="multifeature")
        bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(audio)

        beat_df = pd.DataFrame({'beats':beats})

        return beat_df


    def bpm(path):
        audio = es.MonoLoader(filename=path1)()

        # Compute beat positions and BPM.
        rhythm_extractor = es.RhythmExtractor2013(method="multifeature")
        bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(audio)
        return round(bpm,0)

    def key(path):
        audio_file = path
        # Initialize algorithms we will use.
        loader = ess.MonoLoader(filename=audio_file)
        framecutter = ess.FrameCutter(frameSize=4096, hopSize=2048, silentFrames='noise')
        windowing = ess.Windowing(type='blackmanharris62')
        spectrum = ess.Spectrum()
        spectralpeaks = ess.SpectralPeaks(orderBy='magnitude',
                                          magnitudeThreshold=0.00001,
                                          minFrequency=20,
                                          maxFrequency=3500,
                                          maxPeaks=60)

        hpcp = ess.HPCP()
        hpcp_key = ess.HPCP(size=36, # We will need higher resolution for Key estimation.
                            referenceFrequency=440, # Assume tuning frequency is 44100.
                            bandPreset=False,
                            minFrequency=20,
                            maxFrequency=1500,
                            weightType='cosine',
                            nonLinear=False,
                            windowSize=1.)

        key = ess.Key(profileType='edma', # Use profile for electronic music.
                      numHarmonics=4,
                      pcpSize=36,
                      slope=0.6,
                      usePolyphony=True,
                      useThreeChords=True)

        # Use pool to store data.
        pool = essentia.Pool()

        # Connect streaming algorithms.
        loader.audio >> framecutter.signal
        framecutter.frame >> windowing.frame >> spectrum.frame
        spectrum.spectrum >> spectralpeaks.spectrum
        spectralpeaks.magnitudes >> hpcp.magnitudes
        spectralpeaks.frequencies >> hpcp.frequencies
        spectralpeaks.magnitudes >> hpcp_key.magnitudes
        spectralpeaks.frequencies >> hpcp_key.frequencies
        hpcp_key.hpcp >> key.pcp
        hpcp.hpcp >> (pool, 'tonal.hpcp')
        key.key >> (pool, 'tonal.key_key')
        key.scale >> (pool, 'tonal.key_scale')
        key.strength >> (pool, 'tonal.key_strength')

        essentia.run(loader)

        print("Estimated key and scale:", pool['tonal.key_key'] + " " + pool['tonal.key_scale'])


    def fade_out(sample, start_time, fade_duration):

        dur = librosa.get_duration(y=sample)
        fps = len(sample) / dur

        start_frame = int(start_time * fps)
        end_frame = int((start_time + fade_duration) * fps)

        fading_audio = sample[start_frame:end_frame] 
        env = np.linspace(1,0, num=len(fading_audio))
        fading_audio = fading_audio * env

        full_audio = list(sample[:start_frame]) + list(fading_audio) + list(sample[end_frame:] * np.zeros(len(sample[end_frame:])))

        return np.array(full_audio)


    def fade_in(sample, start_time, fade_duration):
        dur = librosa.get_duration(y=sample)
        fps = len(sample) / dur

        start_frame = int(start_time * fps)
        end_frame = int((start_time + fade_duration) * fps)

        fading_audio = sample[start_frame:end_frame] 
        env = np.linspace(0,1, num=len(fading_audio))
        fading_audio = fading_audio * env

        full_audio =list(fading_audio) + list(sample[end_frame:] * np.ones(len(sample[end_frame:])))

        return np.array(full_audio)


    def speed_control(sample, sr, start_time, end_time, original_bpm, new_bpm):

        speed_rate = (new_bpm/original_bpm)
        speed_frags = abs((1-speed_rate) / 15)
        time_rates = []
        start_rate = 1

        for i in range(15):
            if speed_rate >= 1:
                start_rate += speed_frags
                time_rates.append(start_rate)

            if speed_rate < 1:
                start_rate -= speed_frags
                time_rates.append(start_rate)

        y_new = []

        speed_sample = sample[int(start_time*sr): int(end_time*sr)]
        frame_size = len(speed_sample) / 15

        before_sample = sample[:int(start_time*sr)]
        after_sample = pyrb.time_stretch(sample[int(end_time*sr):], sr=sr, rate=speed_rate)

        pct = (end_time - start_time )/ 15

        start = 0
        end = 15

        for i,j in zip(range(15),time_rates):

                yn = speed_sample[int(i*frame_size): int((i+1)*frame_size)]
                yn = pyrb.time_stretch(yn, sr=sr,rate=j)
                yn = fade(yn, .003, .003)

                y_new += (list(yn))


        y_new_final = np.array(y_new)
        y_new_final = np.concatenate((before_sample, y_new_final, after_sample))

        return y_new_final


    def highpass_control(sample, sr, start_time, end_time, cutoff_freq, order):

        high_sample = sample[int(start_time*sr):int(end_time*sr)]
        before_sample = sample[:int(start_time*sr)]
        after_sample = sample[int(end_time*sr):]

        cutoff_log = np.log10(cutoff_freq)
        log_intervals = np.logspace(1.7, cutoff_log, 50)
        frame_size = len(high_sample) / 50
        y_new = []

        for i,j in zip(range(50),log_intervals):

            yn = high_sample[int(i*frame_size): int((i+1)*frame_size)]

            nyquist_freq = 0.5 * sr
            normalized_cutoff_freq = j / nyquist_freq
            b, a = scipy.signal.butter(order, normalized_cutoff_freq, btype='high', analog=False)

            yn = scipy.signal.lfilter(b, a, yn)
            yn = fade(yn, .003, .003)
            y_new += (list(yn))

        nyquist_freq = 0.5 * sr
        normalized_cutoff_freq = log_intervals[-1] / nyquist_freq
        b, a = scipy.signal.butter(order, normalized_cutoff_freq, btype='high', analog=False)
        after_sample = scipy.signal.lfilter(b, a, after_sample)

        y_new_array = np.array(y_new)
        y_new_final = np.concatenate((before_sample, y_new_array, after_sample))

        return y_new_final

    def lowpass_control(sample, sr, start_time, end_time, cutoff_freq, order):

        high_sample = sample[int(start_time*sr):int(end_time*sr)]
        before_sample = sample[:int(start_time*sr)]
        after_sample = sample[int(end_time*sr):]

        cutoff_log = np.log10(cutoff_freq)
        log_intervals = np.logspace(4, cutoff_log, 50)
        frame_size = len(high_sample) / 50
        y_new = []

        for i,j in zip(range(50),log_intervals):

            yn = high_sample[int(i*frame_size): int((i+1)*frame_size)]

            nyquist_freq = 0.5 * sr
            normalized_cutoff_freq = j / nyquist_freq
            b, a = scipy.signal.butter(order, normalized_cutoff_freq, btype='high', analog=False)

            yn = scipy.signal.lfilter(b, a, yn)
            yn = fade(yn, .003, .003)
            y_new += (list(yn))

        nyquist_freq = 0.5 * sr
        normalized_cutoff_freq = log_intervals[-1] / nyquist_freq
        b, a = scipy.signal.butter(order, normalized_cutoff_freq, btype='high', analog=False)
        after_sample = scipy.signal.lfilter(b, a, after_sample)

        y_new_array = np.array(y_new)
        y_new_final = np.concatenate((before_sample, y_new_array, after_sample))

        return y_new_final